# Person 1 — Logistic Regression & Data Collection Pipeline

Pipeline Responsibility: Data Collection & Inventory Audit\nModel Assignment: Logistic Regression

In [1]:
from pathlib import Path
import json, sys, time
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import joblib

HERE = Path.cwd().resolve()
ROOT = next((p for p in (HERE, *HERE.parents) if (p / "data").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Could not find project root.")

ARTIFACTS = ROOT / "parts" / "artifacts"
OUTPUT_DIR = HERE / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42

manifest = pd.read_csv(ARTIFACTS / "02_clean_manifest.csv")
features_data = np.load(ARTIFACTS / "03_features.npz")
X = features_data["X"]
y = manifest["label"].to_numpy()

dev_mask = manifest["split"] == "development"
test_mask = manifest["split"] == "test"

X_tr, y_tr = X[dev_mask], y[dev_mask]
X_te, y_te = X[test_mask], y[test_mask]

print(f"Loaded {len(X_tr)} training samples and {len(X_te)} test samples.")


Loaded 717 training samples and 180 test samples.


In [2]:
from sklearn.linear_model import LogisticRegression

print("--- Person 1: Logistic Regression Model Training & Data Inventory ---")
model = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=3000, class_weight='balanced', random_state=SEED))

start_time = time.perf_counter()
model.fit(X_tr, y_tr)
fit_time = time.perf_counter() - start_time

preds = model.predict(X_te)
acc = accuracy_score(y_te, preds)
p, r, f1, _ = precision_recall_fscore_support(y_te, preds, average='macro')
cm = confusion_matrix(y_te, preds, labels=["Healthy", "Unhealthy"])

print(f"Accuracy: {acc:.4f}")
print(f"Macro F1: {f1:.4f}")
print("Confusion Matrix:\n", cm)
print("\nClassification Report:\n", classification_report(y_te, preds))

# Inspect top Logistic Regression coefficients
clf = model.named_steps['logisticregression']
coefs = clf.coef_[0]
top_healthy_idx = np.argsort(coefs)[:10]
top_unhealthy_idx = np.argsort(coefs)[-10:][::-1]

print("Top 10 features favoring Healthy class:", top_healthy_idx)
print("Top 10 features favoring Unhealthy class:", top_unhealthy_idx)

metrics = {
    "model_name": "Logistic Regression",
    "pipeline_stage": "Data Collection & Inventory",
    "accuracy": float(acc),
    "macro_f1": float(f1),
    "precision": float(p),
    "recall": float(r),
    "fit_time_seconds": float(fit_time),
    "confusion_matrix": cm.tolist(),
    "classes": ["Healthy", "Unhealthy"]
}

with open(OUTPUT_DIR / "logistic_regression_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

joblib.dump(model, OUTPUT_DIR / "logistic_regression_model.joblib")
print("Saved outputs to:", OUTPUT_DIR)


--- Person 1: Logistic Regression Model Training & Data Inventory ---


Accuracy: 0.9222
Macro F1: 0.9222
Confusion Matrix:
 [[84  8]
 [ 6 82]]

Classification Report:
               precision    recall  f1-score   support

     Healthy       0.93      0.91      0.92        92
   Unhealthy       0.91      0.93      0.92        88

    accuracy                           0.92       180
   macro avg       0.92      0.92      0.92       180
weighted avg       0.92      0.92      0.92       180

Top 10 features favoring Healthy class: [ 69  68  57  67  66 111  54 369  65  20]
Top 10 features favoring Unhealthy class: [ 55 106  61  84  60  13  95  25  24  59]
Saved outputs to: D:\SLIIT\projectr\Dataset_Train\parts\logistic_regression\outputs
